# Build the oceans shapefile

Generate a shapefile with all the oceans and seas grouped as follows:

| Ocean / Sea                              |
|:-----------------------------------------|
| Southern Ocean                           |
| Southern Atlantic Ocean                  |
| South Pacific Ocean                      |
| North Pacific Ocean                      |
| South China and Easter Archipelagic Seas |
| Indian Ocean                             |
| Mediterranean Region                     |
| Baltic Sea                               |
| North Atlantic Ocean                     |
| Arctic Ocean Caspian Sea                 |


### Input data:


**GOaS and SeaVox** @  https://www.marineregions.org/downloads.php:
   - Global Oceans and Seas, version 1 (Flanders Marine Institute, 2021) https://doi.org/10.14284/542 for the Oceans
   - Polygon dataset of the extent of water bodies from the SeaVoX Salt and Fresh Water Body Gazetteer (v19) (British Oceanographic Data Centre, 2023) https://doi.org/10.14284/590 for the Caspian Sea

### Input data license:

Global Oceans and Seas, version 1 (Flanders Marine Institute, 2021): CC BY 4.0 and Polygon dataset of the extent of water bodies from the SeaVoX Salt and Fresh Water Body Gazetteer (v19) (British Oceanographic Data Centre, 2023): CC BY 4.0

Giulia Cigna - giulia.cigna@polit.it<br>
Romain Thomas - romain.thomas@polito.it<br>
2026

In [1]:
import os
from dotenv import load_dotenv
import geopandas as gpd
import pandas as pd
import logging
from pathlib import Path
import chardet

## LOGGING

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True
)

## SETTINGS

In [3]:
if not os.path.exists(".env"):
    raise ValueError("You must create the '.env' file and set the values before running this notebook.")

load_dotenv()

True

## OUTPUT SETTINGS

In [4]:
# Get full shapefile path from environment
oceans_from_wb_path = os.getenv("OCEANS_FROM_WB_PATH")
if oceans_from_wb_path is None:
    raise ValueError("OCEANS_FROM_WB_PATH environment variable is not set")

# Create parent directory if it doesn't exist
Path(oceans_from_wb_path).parent.mkdir(parents=True, exist_ok=True)

logging.info(f"Output path: {oceans_from_wb_path}")


2026-03-17 18:11:50 - root - INFO - Output path: data/ocean_from_wb/ocean_from_wb.shp


## INPUT FILES

In [5]:
# shapefile
wb_oceans_path = os.getenv("WB_OCEAN_PATH")
if wb_oceans_path is None:
    raise ValueError("WB_OCEAN_PATH environment variable is not set")


# CSV with regions ID
codes_id_path = os.getenv('CODES_ID_PATH')
if codes_id_path is None:
    raise ValueError("CODES_ID_PATH environment variable is not set")


## READING INPUT FILES

In [6]:
logging.info(f"Reading shapefile for Ocean Mak from {wb_oceans_path}")
gdf_oceans_wb = gpd.read_file(wb_oceans_path)


2026-03-17 18:11:50 - root - INFO - Reading shapefile for Ocean Mak from data/World Bank Official Boundaries - Ocean Mask - apr2025/WB_GAD_ocean_mask.shp


In [7]:
# regions ID
logging.info(f"Reading ids from {codes_id_path}")
with open(codes_id_path, "rb") as f:
    result = chardet.detect(f.read())

# from https://pandas.pydata.org/pandas-docs/stable/user_guide/io.html#na-values
na_vals = ['-1.#IND', '1.#QNAN', '1.#IND', '-1.#QNAN', '#N/A N/A', '#N/A', 'N/A', 'n/a', 'NA', '<NA>', '#NA', 'NULL', 'null', 'NaN', '-NaN', 'nan', '-nan', 'None', '']
# avoids errors with country code "NA":
na_vals.remove('NA')

codes_id = pd.read_csv(
    codes_id_path,
    encoding=result["encoding"],
    sep=None,
    engine="python",
    keep_default_na=False,
    na_values=na_vals
)


2026-03-17 18:11:54 - root - INFO - Reading ids from ../data/codes_id.csv


## METADATA


In [8]:
# Scheme definition of the final shape file

schema = {
    "ID": "str:10",
    "NAME": "str:50",
    "ISO3_CODE": "str:3",
    "ISO2_CODE": "str:2",
    "ISON_CODE": "int",
    "NUM_ID": "int",
    'WB_A3': "str:3",
    'HASC_0': "str:3",
    'WB_REGION': "str:50",
    'WB_STATUS': "str:50",
    'SOVEREIGN': "str:50",
    "SOURCE": "str:50",
    "geometry": "MultiPolygon"
}


In [9]:
# Merge together the gdfs
gdf_final = gpd.GeoDataFrame(
    pd.concat([gdf_oceans_wb], ignore_index=True),
    crs="EPSG:4326"
)

Adding Ocean Metadata

In [10]:

gdf_final["ID"] = "OC_AQ"
gdf_final["NAME"] = "Ocean with Antarctica"


In [11]:
# Add source
gdf_final["SOURCE"] = "World Bank Official Boundaries"

## SAVING OUTPUT FILE

In [12]:

# Add the NUM_ID from the csv file
lookup = codes_id.set_index("ID")["NUM_ID"]
gdf_final["NUM_ID"] = gdf_final["ID"].map(lookup)
gdf_final["NUM_ID"] = gdf_final["NUM_ID"].astype(str).str.strip()
gdf_final["NUM_ID"] = pd.to_numeric(gdf_final["NUM_ID"], errors='coerce').astype('Int64')


In [13]:
gdf_final = gdf_final.drop(columns='id')

In [14]:
# Save GeoDataFrame
gdf_final.to_file(oceans_from_wb_path)

logging.info(f"Saved Shapefile to: {oceans_from_wb_path}")

2026-03-17 18:11:55 - pyogrio._io - INFO - Created 1 records
2026-03-17 18:11:55 - root - INFO - Saved Shapefile to: data/ocean_from_wb/ocean_from_wb.shp
